# 02 — Resolution Matching

**Purpose**: Match each resolved Polymarket equity market to its actual stock price outcome.

This is the **core dataset** for the thesis. For each closed market, we determine:
- Was the predicted event true ("Yes") or false ("No")?
- What was the final market price (implied probability)?

**Outputs**: `data/market_resolutions.parquet`

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import config as cfg
import thesis_utils as tu

data_dir, results_dir = tu.ensure_project_dirs(cfg.PROJECT_ROOT)

## 1. Load Inputs

In [ ]:
# Load the classified market inventory from 00_data_audit
inv_path = results_dir / 'full_market_inventory.csv'
df_inv = pd.read_csv(inv_path)
print(f'Market inventory: {len(df_inv):,} markets')

# Load stock prices from 01_data_collection
stock_prices = tu.load_parquet('stock_prices_daily.parquet')
stock_prices.index = pd.to_datetime(stock_prices.index)
print(f'Stock prices: {stock_prices.shape[0]:,} days, {stock_prices.shape[1]} columns')
print(f'Date range: {stock_prices.index.min().date()} to {stock_prices.index.max().date()}')

# Load CLOB prices (for implied probability)
df_clob = tu.load_parquet('clob_prices_snapshot.parquet', required=False)
if df_clob is not None:
    print(f'CLOB prices: {df_clob["clob_price"].notna().sum():,} valid')

## 2. Filter to Resolvable Markets

In [ ]:
resolvable_types = {'daily_updown', 'daily_close_above', 'daily_close_range',
                    'weekly_above', 'weekly_range', 'monthly_above',
                    'monthly_hit', 'yearly_range'}

df = df_inv[
    df_inv['closed'] &
    df_inv['ticker'].notna() &
    df_inv['resolution_date'].notna() &
    df_inv['market_type'].isin(resolvable_types)
].copy()

df['resolution_date'] = pd.to_datetime(df['resolution_date'])
df['volume_num'] = pd.to_numeric(df['volume'], errors='coerce')

print(f'Resolvable closed markets: {len(df):,}')
print(f'\nBy type:')
print(df['market_type'].value_counts())
print(f'\nBy ticker (top 10):')
print(df['ticker'].value_counts().head(10))

## 3. Resolve Each Market

In [ ]:
# Resolve outcomes against actual stock prices
outcomes = []
for idx, row in df.iterrows():
    outcome = tu.resolve_market(row, stock_prices)
    outcomes.append(outcome)

df['outcome'] = outcomes
df['outcome_int'] = df['outcome'].apply(lambda x: int(x) if x is not None else np.nan)

resolved = df['outcome'].notna().sum()
print(f'Successfully resolved: {resolved:,} / {len(df):,} ({resolved/len(df)*100:.1f}%)')
print(f'\nOutcome distribution:')
print(df['outcome'].value_counts(dropna=False))

# Base rate
valid = df[df['outcome'].notna()]
yes_rate = valid['outcome_int'].mean()
print(f'\nBase rate (P(Yes)): {yes_rate:.3f}')

In [ ]:
# Why did some fail to resolve?
unresolved = df[df['outcome'].isna()]
print(f'Unresolved: {len(unresolved):,}')
if len(unresolved) > 0:
    print(f'\nBy type:')
    print(unresolved['market_type'].value_counts())
    print(f'\nBy ticker:')
    print(unresolved['ticker'].value_counts().head(10))
    print(f'\nSample unresolved:')
    for q in unresolved['question'].sample(min(10, len(unresolved)), random_state=42):
        print(f'  - {q}')

## 4. Merge CLOB Prices (Implied Probability)

In [ ]:
# Merge CLOB price as implied probability
if df_clob is not None:
    df = df.merge(df_clob[['market_id', 'clob_price']], on='market_id', how='left')
    valid_prices = df['clob_price'].notna().sum()
    print(f'Markets with CLOB implied probability: {valid_prices:,} / {len(df):,}')
else:
    df['clob_price'] = np.nan
    print('No CLOB prices available — will need to use alternative implied probability source')

# For markets without CLOB price, we can potentially use the volume-weighted estimate
# or mark them as needing the Gamma API midpoint
print(f'\nMarkets with both outcome AND implied probability: {(df["outcome"].notna() & df["clob_price"].notna()).sum():,}')

## 5. Fallback: Gamma API for Final Prices

For closed markets without CLOB prices, try the Gamma API which may have the resolved outcome recorded.

In [ ]:
import requests, time

def fetch_gamma_market(market_id: str) -> dict | None:
    """Fetch market details from Gamma API."""
    url = f'{cfg.GAMMA_BASE}/markets/{market_id}'
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            return r.json()
    except Exception:
        pass
    return None

# Fetch for markets missing CLOB price but with valid outcome
need_price = df[(df['outcome'].notna()) & (df['clob_price'].isna())].copy()
print(f'Markets needing Gamma API price: {len(need_price):,}')

gamma_prices = {}
for i, (idx, row) in enumerate(need_price.iterrows()):
    if i % 100 == 0 and i > 0:
        print(f'  Fetched {i}/{len(need_price)}...')
    result = fetch_gamma_market(row['market_id'])
    if result:
        # Gamma API returns 'outcomePrices' for resolved markets
        prices = result.get('outcomePrices')
        if prices:
            try:
                # outcomePrices is a JSON string like '["0.95","0.05"]'
                import json
                price_list = json.loads(prices) if isinstance(prices, str) else prices
                gamma_prices[row['market_id']] = float(price_list[0])  # Yes price
            except Exception:
                pass
    time.sleep(0.15)

print(f'\nGot {len(gamma_prices):,} prices from Gamma API')

In [ ]:
# Merge Gamma prices into the main DataFrame
if gamma_prices:
    df['gamma_price'] = df['market_id'].map(gamma_prices)
    # Use gamma_price where clob_price is missing
    df['implied_prob'] = df['clob_price'].fillna(df.get('gamma_price', np.nan))
else:
    df['implied_prob'] = df['clob_price']

# For resolved markets, the final price should be near 0 or 1
# If we have a resolved outcome but no price, use the Gamma resolved price
# which should be ~1.0 for Yes and ~0.0 for No
final_count = (df['outcome'].notna() & df['implied_prob'].notna()).sum()
print(f'Markets with outcome AND implied probability: {final_count:,}')

# For calibration, we need the LAST TRADING price (before resolution), not the resolved price
# Gamma's outcomePrices after resolution are typically 0.95/0.05 or 1.0/0.0
# These are post-resolution prices and not useful for calibration
# We need the pre-resolution implied probability
print('\nNote: For proper calibration analysis, we need pre-resolution prices.')
print('Markets where implied_prob is close to 0 or 1 are likely post-resolution settled prices.')
if df['implied_prob'].notna().sum() > 0:
    extreme = ((df['implied_prob'] > 0.95) | (df['implied_prob'] < 0.05)).sum()
    print(f'Markets with extreme prices (>0.95 or <0.05): {extreme:,}')

## 6. Quality Checks

In [ ]:
# Final resolved dataset stats
df_resolved = df[df['outcome'].notna()].copy()

print(f'=== Resolution Matching Summary ===')
print(f'Total closed markets processed: {len(df):,}')
print(f'Successfully resolved: {len(df_resolved):,} ({len(df_resolved)/len(df)*100:.1f}%)')
print(f'With implied probability: {df_resolved["implied_prob"].notna().sum():,}')

print(f'\nOutcome distribution:')
print(df_resolved['outcome_int'].value_counts().rename({0: 'No', 1: 'Yes'}))

print(f'\nBy market type:')
type_summary = df_resolved.groupby('market_type').agg(
    n=('outcome_int', 'size'),
    yes_rate=('outcome_int', 'mean'),
    avg_volume=('volume_num', 'mean'),
).sort_values('n', ascending=False)
print(type_summary.to_string())

print(f'\nBy ticker:')
ticker_summary = df_resolved.groupby('ticker').agg(
    n=('outcome_int', 'size'),
    yes_rate=('outcome_int', 'mean'),
).sort_values('n', ascending=False)
print(ticker_summary.head(10).to_string())

In [ ]:
# Spot-check: verify a few resolutions manually
spot_checks = df_resolved.sample(5, random_state=42)
for _, row in spot_checks.iterrows():
    ticker = row['ticker']
    res_date = pd.Timestamp(row['resolution_date'])
    close_col = f'{ticker}_Close'
    if close_col in stock_prices.columns:
        valid_dates = stock_prices.index[stock_prices.index <= res_date]
        if len(valid_dates) > 0:
            actual_date = valid_dates[-1]
            close = stock_prices.loc[actual_date, close_col]
            prev_dates = stock_prices.index[stock_prices.index < actual_date]
            prev_close = stock_prices.loc[prev_dates[-1], close_col] if len(prev_dates) > 0 else None
            print(f'Q: {row["question"][:80]}')
            print(f'  Type: {row["market_type"]}, Strike: {row["strike"]}, Range: [{row["strike_low"]}, {row["strike_high"]}]')
            print(f'  Date: {actual_date.date()}, Close: ${close:.2f}, Prev: ${prev_close:.2f}' if prev_close else f'  Date: {actual_date.date()}, Close: ${close:.2f}')
            print(f'  Resolved: {"Yes" if row["outcome"] else "No"}')
            print()

## 7. Save Core Dataset

In [ ]:
# Select columns for the final resolution dataset
output_cols = [
    'market_id', 'event_id', 'question', 'slug', 'ticker',
    'market_type', 'resolution_date', 'strike', 'strike_low', 'strike_high',
    'volume_num', 'startDate', 'endDate',
    'implied_prob', 'outcome', 'outcome_int',
]
# Only include columns that exist
output_cols = [c for c in output_cols if c in df_resolved.columns]

df_out = df_resolved[output_cols].copy()
df_out = df_out.rename(columns={'volume_num': 'volume'})

out_path = data_dir / 'market_resolutions.parquet'
tu.save_parquet(df_out, out_path)
print(f'Saved {len(df_out):,} resolved markets to {out_path}')
print(f'Columns: {list(df_out.columns)}')